In [ ]:
import chromadb  # Importing the ChromaDB library for vector database operations
from sentence_transformers import SentenceTransformer  # Importing SentenceTransformer for embedding text
import ollama  # Importing the Ollama library to interact with language models

# --- Configuration Section ---
# This section sets up all the configurations needed for the chatbot.
CHROMA_PATH = r"C:\Users\jatro\Dev\Dataframes\chroma_db"  # Path to the ChromaDB database on your local file system.
                                                         # 'r' prefix ensures it's treated as a raw string, preventing issues with backslashes in Windows paths.
COLLECTION_NAME = "conversations"  # Name of the ChromaDB collection where your Airbnb conversation embeddings are stored.
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"  # Name of the Sentence Transformer model used for embedding text.
                                              # 'all-mpnet-base-v2' is a good general-purpose model known for quality embeddings.
MODEL_NAME = "llama3-chatqa:8b"  # Name of the Ollama language model to be used for generating chatbot responses.
                                   # 'llama3-chatqa:8b' is chosen for its question-answering and chat capabilities.
TEMPERATURE = 0.2  # Temperature parameter controls the randomness of the language model's output.
                   # Lower values (e.g., 0.2) make the output more deterministic and focused, less random.
                   # Higher values (closer to 1) make the output more creative and random, but potentially less coherent.

# --- Load Embedding Model ---
# Initialize the Sentence Transformer model. This model will be used to create vector embeddings of text.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
# The SentenceTransformer model is loaded only once at the start, as it can be reused for embedding multiple texts.

# --- Load ChromaDB ---
# Initialize a ChromaDB client to interact with the database stored at CHROMA_PATH.
client = chromadb.PersistentClient(path=CHROMA_PATH)
# Access or create a ChromaDB collection with the name specified in COLLECTION_NAME.
# Collections in ChromaDB are where your embeddings and associated data (like Airbnb responses) are stored.
collection = client.get_collection(name=COLLECTION_NAME)

# Confirmation message to the console, indicating that the chatbot is ready and which model is being used.
print(f"Airbnb Host Chatbot is ready! Using Ollama model '{MODEL_NAME}' with temperature {TEMPERATURE}. Type 'exit' to quit.")

# --- Function to Retrieve Relevant Responses from ChromaDB ---
def get_relevant_responses(user_input, collection, embedding_model, top_n=3):
    """
    Retrieves the most relevant Airbnb responses from the ChromaDB collection based on the user's input.

    Args:
        user_input (str): The user's message or question.
        collection (chromadb.Collection): The ChromaDB collection to query.
        embedding_model (SentenceTransformer): The Sentence Transformer model to embed the user input.
        top_n (int): The number of top relevant responses to retrieve from the database. Default is 3.

    Returns:
        list: A list of the top_n most relevant response texts (documents) from ChromaDB.
    """
    # Embed the user's input message using the Sentence Transformer model.
    # This converts the text into a vector representation in embedding space.
    input_embedding = embedding_model.encode(user_input).tolist()

    # Query the ChromaDB collection to find documents that are semantically similar to the user input embedding.
    results = collection.query(
        query_embeddings=[input_embedding],  # The embedding of the user's query.
        n_results=top_n,  # Number of results to retrieve, as specified by top_n.
        include=["documents"]  # Include the actual text content ('documents') of the retrieved responses in the results.
                               # We assume your Airbnb responses are stored in the 'documents' field in ChromaDB.
    )
    # Extract the actual response texts (documents) from the query results.
    # ChromaDB query results are structured, and 'documents' contains the text content.
    relevant_responses = results['documents'][0] # Access the documents from the first (and only) query in the batch
    return relevant_responses # Return the list of relevant response texts.

# --- Function to Generate Chatbot Response using Ollama and Llama 2 ---
def generate_chatbot_response(user_input, relevant_responses, conversation_history):
    """
    Generates a chatbot response using the Ollama language model, incorporating relevant Airbnb responses as context
    and maintaining conversation history.

    Args:
        user_input (str): The user's current message or question.
        relevant_responses (list): A list of relevant Airbnb response texts retrieved from ChromaDB.
        conversation_history (list): A list of dictionaries representing the history of the conversation,
                                     where each dictionary has 'role' ('user' or 'assistant') and 'content'.

    Returns:
        str: The chatbot's generated response.
    """
    # Join the relevant Airbnb responses into a single context string.
    # Each response is formatted with a hyphen for better readability in the prompt.
    context = "\n".join(f"- {response}" for response in relevant_responses)

    # Initialize an empty string to store the formatted conversation history.
    history_text = ""
    # Check if there's any conversation history.
    if conversation_history:
        history_text = "\nConversation History:\n" # Add a heading for the history in the prompt.
        # Iterate through each turn in the conversation history.
        for turn in conversation_history:
            # Append each turn to the history text, formatting it with the role (user or assistant) and content.
            history_text += f"{turn['role']}: {turn['content']}\n"

    # Construct the prompt to be sent to the language model.
    # This prompt includes system instructions, conversation history, context from ChromaDB, and the user's question.
    prompt = f"""
[INST] <<SYS>>
You are a helpful and friendly Airbnb host chatbot. You should provide accurate and reliable responses.
Use the following Airbnb host responses as context to answer user questions about hosting.
If the context is not directly relevant, answer based on your general knowledge of hosting on Airbnb in a friendly and helpful tone, while still being accurate.
Refer to the conversation history to maintain context and provide coherent answers.

{history_text} # Insert the formatted conversation history into the prompt.
<</SYS>>

Context:
{context} # Insert the relevant Airbnb responses as context.

User question: {user_input} [/INST] # Insert the user's current question.
"""
    # Send the prompt to the Ollama chat API to get a response from the language model.
    ollama_response = ollama.chat(
        model=MODEL_NAME,  # Specify the language model to use (defined in configuration).
        messages=[
            {'role': 'system', 'content': 'You are a helpful and friendly Airbnb host chatbot. You should provide accurate and reliable responses. Use Airbnb host responses as context and conversation history.'}, # Define system role and behavior more explicitly
            *conversation_history,  # Include the entire conversation history as messages for context.
                                    # The '*' operator unpacks the list of history dictionaries into individual arguments.
            {'role': 'user', 'content': prompt}  # The main user prompt containing instructions, context, and user question.
        ],
        options={"temperature": TEMPERATURE}  # Set the temperature for response generation (defined in configuration).
                                              # Lower temperature makes responses more deterministic.
    )

    # Extract the text content of the chatbot's response from the Ollama response object.
    chatbot_response = ollama_response['message']['content']
    return chatbot_response # Return the generated chatbot response.

# --- Main execution block ---
if __name__ == "__main__":
    # Print a message to indicate that the chatbot is ready to use.
    print(f"Airbnb Host Chatbot is ready! Using Ollama model '{MODEL_NAME}' with temperature {TEMPERATURE}. Type 'exit' to quit.")
    conversation_history = [] # Initialize an empty list to store the conversation history for each session.

    # Start an infinite loop to continuously interact with the user until they type 'exit'.
    while True:
        user_message = input("You: ") # Get user input from the console.
        if user_message.lower() == "exit": # Check if the user wants to exit the chat.
            break # If user types 'exit', break out of the loop and end the program.

        # Retrieve relevant Airbnb responses from ChromaDB based on the user's message.
        relevant_responses = get_relevant_responses(user_message, collection, embedding_model)

        # Add the user's message to the conversation history.
        # This keeps track of the conversation flow for contextual responses in subsequent turns.
        conversation_history.append({'role': 'user', 'content': user_message})

        # Generate the chatbot's response using the language model, relevant responses, and conversation history.
        chatbot_response = generate_chatbot_response(user_message, relevant_responses, conversation_history)

        # Add the chatbot's generated response to the conversation history.
        conversation_history.append({'role': 'assistant', 'content': chatbot_response})

        # Print the chatbot's response to the console for the user to see.
        print(f"Chatbot: {chatbot_response}")